# Ponto de Controle
Este notebook valida e escreve dados transformados no Google Sheets.

In [1]:
import os
import pandas as pd
import datetime as dt
from extract import read_df
from treat.utils.datas import normalize_date_to_str_DD_M_YYYY
from treat.utils.write_dataframe_to_sheet import write_dataframe_to_sheet
from treat.utils.renomeacoes import renomear_colunas_origem_para_modelo as rename
from treat.utils.normalize import normalize_vehicle
from treat.utils.datas import concat_period
from treat.utils.campos_calculados import make_id_ponto_de_controle

In [2]:
# Flags de execução
"""
Célula  – Imports & parâmetros globais

Define:
- Módulos padrão e helpers do projeto
- Flags de execução e IDs de planilhas via env var
- Constantes de aba, cabeçalho e filtro de data
- Lista de colunas de destino
"""
DRY_RUN = True

# IDs das planilhas via variáveis de ambiente
os.environ["ORIGIN_SHEET_ID"] = "1DazUQxspLgT0utOFHcTINbFngXw7Fq0LOq6v4lRGixg"
ORIGIN_SHEET_ID = os.getenv("ORIGIN_SHEET_ID")
os.environ["DEST_SHEET_ID"] = "1DpH5tu4KJKqbA6ueFtf1s1FueBkR4-EtPf5xHyXx8zw"
DEST_SHEET_ID   = os.getenv("DEST_SHEET_ID")

# Garantia de que foram definidas
assert ORIGIN_SHEET_ID is not None and DEST_SHEET_ID is not None, \
    "Defina as variáveis de ambiente ORIGIN_SHEET_ID e DEST_SHEET_ID"


In [3]:
# Constantes de aba & cabeçalho: nomes centralizados em um só lugar
ORIGIN_TAB    = "modeloGeral"
DEST_TAB      = "IMPULSIONAMENTOS 2025"
HEAD_ROW_DEST = 4  # zero-based (header na linha 5)



In [4]:
# Filtro temporal & Data mínima – usado no filtro posterior
MIN_DATE = dt.date(2025, 6, 1)


In [5]:
# Lista DEST_COLUMNS: declara lista com 11 colunas, ordem exata exigida
DEST_COLUMNS = [
    "Data",
    "Campanha",
    "Veículo",
    "Link conteúdos impulsionados",
    "Período",
    "Agência",
    "Editoria",
    "Objetivo",
    "Meta",
    "Status",
    "Resultado",
]
assert len(DEST_COLUMNS) == 11, f"DEST_COLUMNS deve ter 11 colunas, mas tem {len(DEST_COLUMNS)}"
    

In [6]:
# Leitura da aba de origem – deve executar sem exceção se ORIGIN_SHEET_ID estiver definido
"""
Célula 2 – Leitura + filtro temporal da aba modeloGeral
- Lê df_origin com read_df()
- Converte coluna date para date_dt
- Filtra linhas >= MIN_DATE
- Garante colunas críticas e prepara df_origin
"""
df_origin = read_df(
    sheet_id=ORIGIN_SHEET_ID,
    tab=ORIGIN_TAB,
    header_row=0,
)

In [7]:
# Sanitizar tipos de data – converte coluna 'date' para datetime.date e força dtype str para preservar zeros
df_origin['date'] = df_origin['date'].astype(str)
df_origin['date_dt'] = pd.to_datetime(df_origin['date'], errors='coerce').dt.date

# Preview das datas para validação
if DRY_RUN:
    display(df_origin[['date', 'date_dt']].head())
    invalidados = df_origin['date_dt'].isna().sum()
    print(f"Valores inválidos ou não parseados: {invalidados}")


,date,date_dt
0,2025-03-01,2025-03-01
1,2025-03-01,2025-03-01
2,2025-03-02,2025-03-02
3,2025-03-02,2025-03-02
4,2025-03-03,2025-03-03


Valores inválidos ou não parseados: 0


In [8]:
# Quick-preview em DRY_RUN – exibe head e contagem somente em Dry Run
if DRY_RUN:
    display(df_origin.head())
    print(f"Total de linhas em df_origin: {len(df_origin)}")


,date,account_name,Campanha,ID_Campanha,Veiculo,ID_Veiculo,ad_group_name,ad_name,start,end,...,video_watched_25,video_watched_50,video_watched_75,video_watched_100,post_reactions,post_shares,post_comments,Engajamento_Total,ID,date_dt
0,2025-03-01,Sebrae Nacional - DEBRITO,Catalisa ICT,dbt_sbrae_2025_catalisa,Instagram,2,2025_2_BR_TRAF_CPC_AS 25+ MD. DR,2025_2_BR_VÍDEO_GABI_BAILAS_ACAO_DBT_SBRAE_202...,2025-02-24,2025-03-09,...,7641,4014,291,24,3,21,0,24,"2025-03-01-Catalisa ICT-144387-1914,79-3349",2025-03-01
1,2025-03-01,Sebrae Nacional - DEBRITO,Catalisa ICT,dbt_sbrae_2025_catalisa,Instagram,2,2025_2_BR_TRAF_CPC_AS 25+ MD. DR,2025_2_BR_VÍDEO_MARI_KRUGER_ACAO_DBT_SBRAE_202...,2025-02-24,2025-03-09,...,9856,1339,148,118,41,31,0,72,"2025-03-01-Catalisa ICT-173584-1950,14-3344",2025-03-01
2,2025-03-02,Sebrae Nacional - DEBRITO,Catalisa ICT,dbt_sbrae_2025_catalisa,Instagram,2,2025_2_BR_TRAF_CPC_AS 25+ MD. DR,2025_2_BR_VÍDEO_GABI_BAILAS_ACAO_DBT_SBRAE_202...,2025-02-24,2025-03-09,...,6380,3159,354,30,3,14,0,17,"2025-03-02-Catalisa ICT-161093-1898,22-3143",2025-03-02
3,2025-03-02,Sebrae Nacional - DEBRITO,Catalisa ICT,dbt_sbrae_2025_catalisa,Instagram,2,2025_2_BR_TRAF_CPC_AS 25+ MD. DR,2025_2_BR_VÍDEO_MARI_KRUGER_ACAO_DBT_SBRAE_202...,2025-02-24,2025-03-09,...,9276,1519,188,111,27,32,0,59,"2025-03-02-Catalisa ICT-219773-2059,91-3270",2025-03-02
4,2025-03-03,Sebrae Nacional - DEBRITO,Catalisa ICT,dbt_sbrae_2025_catalisa,Instagram,2,2025_2_BR_TRAF_CPC_AS 25+ MD. DR,2025_2_BR_VÍDEO_GABI_BAILAS_ACAO_DBT_SBRAE_202...,2025-02-24,2025-03-09,...,6040,2901,330,35,2,10,0,12,"2025-03-03-Catalisa ICT-180854-1849,24-2975",2025-03-03


Total de linhas em df_origin: 1919


In [9]:
# Assert de colunas críticas – garante que df_origin tenha todas as colunas necessárias
required_columns = ["date", "Campanha", "Veiculo", "URL_do_Anuncio", "objective"]
missing_cols = [col for col in required_columns if col not in df_origin.columns]
if missing_cols:
    raise RuntimeError(f"Colunas críticas ausentes em df_origin: {missing_cols}")


In [10]:
# Clean-up da coluna auxiliar – remover 'date_dt' apenas após aplicar o filtro de data mínima (útil para debug)
if not DRY_RUN:
    df_origin.drop(columns=["date_dt"], inplace=True)


In [11]:
# Clonar DataFrame para não mutar a leitura crua
df = df_origin.copy()


In [12]:
# Normalizar coluna Data e dropar coluna date após criar Data para evitar conflito de nomes
df["Data"] = df["date"].apply(normalize_date_to_str_DD_M_YYYY)
df.drop(columns=["date"], inplace=True)

# Debug: mostrar os primeiros valores de 'Data' e conferir dtype
print("Preview 'Data':")
print(df["Data"].head())
print(f"Tipo de coluna Data: {df['Data'].dtype}, total de linhas: {len(df)}")


Preview 'Data':
0    01/3/2025
1    01/3/2025
2    02/3/2025
3    02/3/2025
4    03/3/2025
Name: Data, dtype: object
Tipo de coluna Data: object, total de linhas: 1919


In [ ]:
# Normalizar Veiculo – aplicação de normalize_vehicle
df["Veiculo"] = df["Veiculo"].apply(normalize_vehicle)
df.drop(columns=["Veiculo"], inplace=True)              # opcional: remove a antiga


In [14]:
# Gerar coluna Período com concat_period e dropar colunas auxiliares
df["Período"] = df.apply(lambda r: concat_period(r.get("start"), r.get("end")), axis=1)
df.drop(columns=["start", "end"], inplace=True)

# Debug: conferir se a geração de 'Período' funcionou
print("Preview 'Período':")
print(df["Período"].head().tolist())
non_empty = df["Período"].astype(bool).sum()
print(f"Linhas com período não vazio: {non_empty} de {len(df)}")


Preview 'Período':
['24/2/2025 a 09/3/2025', '24/2/2025 a 09/3/2025', '24/2/2025 a 09/3/2025', '24/2/2025 a 09/3/2025', '24/2/2025 a 09/3/2025']
Linhas com período não vazio: 1919 de 1919


In [15]:
# Mapear colunas diretas
df["Campanha"] = df["Campanha"]
df["Link conteúdos impulsionados"] = df["URL_do_Anuncio"]
df["Objetivo"] = df["objective"]

# Debug prints para verificar mapeamento
print("Preview 'Campanha':", df["Campanha"].head().tolist())
print("Preview 'Link conteúdos impulsionados':", df["Link conteúdos impulsionados"].head().tolist())
print("Preview 'Objetivo",
      df["Objetivo"].head().tolist())

# Asserts para garantir a presença das colunas
assert "Campanha" in df.columns, "Coluna 'Campanha' não encontrada"
assert "Link conteúdos impulsionados" in df.columns, "Coluna 'Link conteúdos impulsionados' não encontrada"
assert "Objetivo" in df.columns, \
       "Coluna 'Objetivo' não encontrada"


Preview 'Campanha': ['Catalisa ICT', 'Catalisa ICT', 'Catalisa ICT', 'Catalisa ICT', 'Catalisa ICT']
Preview 'Link conteúdos impulsionados': ['https://www.instagram.com/p/DGlCKhYA1Tn/', 'https://www.instagram.com/p/DGgVgE6gMQ8/', 'https://www.instagram.com/p/DGlCKhYA1Tn/', 'https://www.instagram.com/p/DGgVgE6gMQ8/', 'https://www.instagram.com/p/DGlCKhYA1Tn/']
Preview 'Objetivo ['Tráfego', 'Tráfego', 'Tráfego', 'Tráfego', 'Tráfego']


In [16]:
# Colunas constantes / vazias
df["Agência"] = "De Brito"
df["Editoria"] = df["Campanha"]
df["Meta"] = ""
df["Status"] = ""
df["Resultado"] = ""

# Debug prints para verificar preenchimento
print("Preview 'Agência':", df["Agência"].head().tolist())
print("Preview 'Editoria':", df["Editoria"].head().tolist())
print("Valores únicos em 'Agência':", df["Agência"].unique())
print("Contagem não vazia em 'Meta (número)':",
      df["Meta"].astype(bool).sum())
print("Contagem não vazia em 'Status':",
      df["Status"].astype(bool).sum())
print("Contagem não vazia em 'Resultado':",
      df["Resultado"].astype(bool).sum())

# Asserts para garantir colunas e conteúdo esperado
assert "Agência" in df.columns and df["Agência"].eq("De Brito").all(), \
    "Erro em 'Agência': valores diferentes de 'De Brito' ou coluna ausente"
assert "Editoria" in df.columns, "Coluna 'Editoria' ausente"
assert all(df["Meta"] == ""), \
    "'Meta"
assert all(df["Status"] == ""), "'Status' deve ser completamente vazio"
assert all(df["Resultado"] == ""), "'Resultado' deve ser completamente vazio"


Preview 'Agência': ['De Brito', 'De Brito', 'De Brito', 'De Brito', 'De Brito']
Preview 'Editoria': ['Catalisa ICT', 'Catalisa ICT', 'Catalisa ICT', 'Catalisa ICT', 'Catalisa ICT']
Valores únicos em 'Agência': ['De Brito']
Contagem não vazia em 'Meta (número)': 0
Contagem não vazia em 'Status': 0
Contagem não vazia em 'Resultado': 0


In [17]:
# Reordenar / reindexar com DEST_COLUMNS e preencher vazios
df_transf = df.reindex(columns=DEST_COLUMNS, fill_value="")

# Debug prints e asserts para verificar ordem e conteúdo das colunas
print("Colunas em df_transf:", df_transf.columns.tolist())
assert df_transf.columns.tolist() == DEST_COLUMNS, (
    f"Colunas fora de ordem ou faltando: {df_transf.columns.tolist()}"
)

# Mostrar as primeiras linhas para confirmação visual
display(df_transf.head())
print(f"Total de linhas em df_transf: {len(df_transf)}")


Colunas em df_transf: ['Data', 'Campanha', 'Veículo', 'Link conteúdos impulsionados', 'Período', 'Agência', 'Editoria', 'Objetivo', 'Meta', 'Status', 'Resultado']


,Data,Campanha,Veículo,Link conteúdos impulsionados,Período,Agência,Editoria,Objetivo,Meta,Status,Resultado
0,01/3/2025,Catalisa ICT,,https://www.instagram.com/p/DGlCKhYA1Tn/,24/2/2025 a 09/3/2025,De Brito,Catalisa ICT,Tráfego,,,
1,01/3/2025,Catalisa ICT,,https://www.instagram.com/p/DGgVgE6gMQ8/,24/2/2025 a 09/3/2025,De Brito,Catalisa ICT,Tráfego,,,
2,02/3/2025,Catalisa ICT,,https://www.instagram.com/p/DGlCKhYA1Tn/,24/2/2025 a 09/3/2025,De Brito,Catalisa ICT,Tráfego,,,
3,02/3/2025,Catalisa ICT,,https://www.instagram.com/p/DGgVgE6gMQ8/,24/2/2025 a 09/3/2025,De Brito,Catalisa ICT,Tráfego,,,
4,03/3/2025,Catalisa ICT,,https://www.instagram.com/p/DGlCKhYA1Tn/,24/2/2025 a 09/3/2025,De Brito,Catalisa ICT,Tráfego,,,


Total de linhas em df_transf: 1919


In [18]:
# 3.7.1 – Dropar colunas auxiliares 'start' e 'end' se ainda existirem
aux_cols = [c for c in ["start", "end"] if c in df_transf.columns]
if aux_cols:
    df_transf.drop(columns=aux_cols, inplace=True)

# Debug: confirmar que as colunas auxiliares foram removidas
print("Colunas após remoção de 'start' e 'end':", df_transf.columns.tolist())


Colunas após remoção de 'start' e 'end': ['Data', 'Campanha', 'Veículo', 'Link conteúdos impulsionados', 'Período', 'Agência', 'Editoria', 'Objetivo', 'Meta', 'Status', 'Resultado']


In [19]:
# ---------------------------------------------------------------------------
# Consolidar aba de destino “IMPULSIONAMENTOS 2025” preservando dropdowns,
# check-boxes e outros valores FORMATADOS.
# ---------------------------------------------------------------------------
import os, re, unicodedata, pandas as pd
from treat.utils.get_google_client import get_google_client

# --- constantes que já devem estar no notebook -----------------------------
# DEST_SHEET_ID, DEST_TAB, HEAD_ROW_DEST (=5), DEST_COLUMNS (11 colunas)

# 1 • Cliente gspread e worksheet ------------------------------------------------
CREDS_PATH = os.getenv("GOOGLE_CREDS_PATH", "creds.json")
gclient    = get_google_client(CREDS_PATH)
ws_dest    = gclient.open_by_key(DEST_SHEET_ID).worksheet(DEST_TAB)

# 2 • Captura da linha de cabeçalho bruta (linha 5 = index 4) -------------------
header_raw = ws_dest.row_values(HEAD_ROW_DEST)          # ex.: ['Data ', 'Campanha', ...]
print("Header bruto:", header_raw)

# 3 • Função de limpeza ---------------------------------------------------------
def _clean(label: str) -> str:
    txt = unicodedata.normalize("NFKD", label or "")
    txt = "".join(c for c in txt if not unicodedata.combining(c))
    txt = txt.replace("\n", " ").strip()
    txt = re.sub(r"\s{2,}", " ", txt)
    return txt

cleaned = [_clean(c) for c in header_raw]
print("Header limpo :", cleaned)

# 4 • Baixa todas as linhas de dados (linha 6 em diante) ------------------------
body = ws_dest.get_values(
    f"A{HEAD_ROW_DEST+1}:K",          # colunas A–K
    value_render_option="FORMATTED_VALUE"
)

# 5 • Normaliza cada linha para ter exatamente 11 colunas -----------------------
max_cols   = len(DEST_COLUMNS)
normalized = [(row + [""] * (max_cols - len(row)))[:max_cols] for row in body]

# 6 • Constrói DataFrame com cabeçalho oficial ---------------------------------
df_dest = pd.DataFrame(normalized, columns=DEST_COLUMNS)

# 7 • Remove linhas totalmente vazias ------------------------------------------
df_dest = df_dest.replace("", pd.NA).dropna(how="all").reset_index(drop=True)

# 8 • Pré-visualização e contagem ----------------------------------------------
print(f"Linhas válidas em df_dest: {len(df_dest)}")
display(df_dest.head())
print(df_dest.apply(lambda c: (c != "") & (c.notna())).sum())

# 9 • Agora é seguro gerar o __ID__ --------------------------------------------
df_dest["__ID__"] = df_dest.apply(make_id_ponto_de_controle, axis=1)


Header bruto: ['', '', '', 'Branco: Sem identificação do período ']
Header limpo : ['', '', '', 'Branco: Sem identificacao do periodo']
Linhas válidas em df_dest: 839


,Data,Campanha,Veículo,Link conteúdos impulsionados,Período,Agência,Editoria,Objetivo,Meta,Status,Resultado
0,Data,Campanha,Veiculo,Link conteúdos impulsionados,Período,Agência,Editoria,Objetivo,Meta,Status,Resultado
1,13/01/2025,Tema do Mês - Janeiro - Abertura de Negócios,IG,https://www.instagram.com/p/DEqHzLrsfA6/#adve...,10/01 a 13/01,Moringa,Tema do Mês,<NA>,23.565 cliques estimados,Pausado,<NA>
2,13/01/2025,Tema do Mês - Janeiro - Abertura de Negócios,FB,https://www.facebook.com/100050484649787/posts...,10/01 a 13/01,Moringa,Tema do Mês,<NA>,23.565 cliques estimados,Pausado,<NA>
3,13/01/2025,Tema do Mês - Janeiro - Abertura de Negócios,TikTok,https://vm.tiktok.com/ZMkaqo2P4/,10/01 a 13/01,Moringa,Tema do Mês,<NA>,23.565 cliques estimados,Pausado,<NA>
4,14/01/2025,Tema do Mês - Janeiro - Abertura de Negócios,IG,https://www.instagram.com/p/DE0X4n4MK-i/#adver...,14/01 a 31/01,Moringa,Tema do Mês,<NA>,23.565 cliques estimados,Finalizado,<NA>


Data                            839
Campanha                        838
Veículo                         839
Link conteúdos impulsionados    815
Período                         838
Agência                         839
Editoria                        727
Objetivo                        789
Meta                            111
Status                          837
Resultado                         1
dtype: int64


In [20]:
# ---------------------------------------------------------------------------
# Gerar coluna de identificador único (__ID__) em df_transf
# Mesmo make_id_ponto_de_controle usado para df_dest
# ---------------------------------------------------------------------------

# 1 · Cria/atualiza a coluna __ID__ em df_transf
df_transf["__ID__"] = df_transf.apply(make_id_ponto_de_controle, axis=1)

# 2 · Validação rápida
print(f"__ID__ gerados em df_transf: {df_transf['__ID__'].nunique()} (únicos) / {len(df_transf)} (linhas)")
assert df_transf["__ID__"].isna().sum() == 0, "Há valores NaN em __ID__ em df_transf — verifique campos vazios"

# 3 · Pré-visualização
display(df_transf.head()[["Data", "Campanha", "Veiculo", "__ID__"]])


__ID__ gerados em df_transf: 1551 (únicos) / 1919 (linhas)


KeyError: "['Veiculo'] not in index"

In [ ]:
# ---------------------------------------------------------------------------
# Deduplicação: filtra apenas as linhas novas em df_transf
# ---------------------------------------------------------------------------

# 1 • Cria o DataFrame somente com os registros cujo __ID__ não está em df_dest
novos = df_transf[~df_transf["__ID__"].isin(df_dest["__ID__"])].copy()

# 2 • Validação rápida: nunca adicionar mais registros do que existem na origem
assert len(novos) <= len(df_transf), (
    f"Erro de deduplicação: len(novos)={len(novos)} maior que len(df_transf)={len(df_transf)}"
)

# 3 • Exibe quantas linhas novas foram identificadas
print(f"Linhas novas após deduplicação: {len(novos)}")

# 4 • Pré-visualização das primeiras entradas novas
display(novos.head())
